# Eksplorasi Fitur TSFEL (Anggota 3)
**Tujuan Notebook ini:**
1. Mengekstrak fitur statistik menggunakan library `tsfel`.
2. Menganalisis masalah umum dari ekstraksi (seperti munculnya nilai `inf` atau `NaN`).
3. Membersihkan fitur-fitur tersebut.
4. Melihat korelasi fitur mana yang paling kuat pengaruhnya terhadap deteksi kerusakan ESP (`Is_Anomaly`).

In [ ]:
import pandas as pd
import numpy as np
import tsfel
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

print("Library berhasil dimuat!")

In [ ]:
print("Memuat dan membersihkan data...")
# Load 1000 baris pertama dari data asli
df = pd.read_csv('../data/raw/dailyData.csv', usecols=['Well_ID', 'ESP Data - Output Frequency', 'ESP Data - Drive Current', 'DOWN_TIME_HOURS'], nrows=1000)
df.columns = ['Well_ID', 'Frequency', 'Load', 'Downtime']

# Interpolasi & Buat Label Target
df['Frequency'] = df['Frequency'].interpolate()
df['Load'] = df['Load'].interpolate()
df.dropna(inplace=True)
df['Is_Anomaly'] = (df['Downtime'] > 0).astype(int)

# Normalisasi 
df['Freq_Norm'] = df['Frequency'] / df['Frequency'].max()
df['Load_Norm'] = df.groupby('Well_ID')['Load'].transform(lambda x: x / x.median())

# Bersihkan inf dari normalisasi awal
df.replace([np.inf, -np.inf], np.nan, inplace=True)
df.fillna(0, inplace=True)

print(f"Data siap! Jumlah sampel: {df.shape[0]}")
print(f"Jumlah Kasus Anomali: {df['Is_Anomaly'].sum()}")

In [ ]:
print("Memulai ekstraksi TSFEL...")

# Kita ambil fitur domain statistik saja agar komputasi ringan
cgf_settings = tsfel.get_features_by_domain('statistical')

# Ekstraksi fitur dari kolom Freq_Norm dan Load_Norm
df_features = tsfel.time_series_features_extractor(
    cgf_settings, 
    df[['Freq_Norm', 'Load_Norm']], 
    window_size=1, 
    verbose=1
)

print(f"\nEkstraksi selesai! TSFEL menghasilkan {df_features.shape[1]} kolom baru.")
df_features.head()

In [ ]:
# Cek nilai Infinity (Tak Terhingga)
inf_counts = np.isinf(df_features).sum().sum()
# Cek nilai NaN (Kosong)
nan_counts = df_features.isna().sum().sum()

print(f"Ditemukan {inf_counts} nilai Infinity (inf) di seluruh data.")
print(f"Ditemukan {nan_counts} nilai kosong (NaN) di seluruh data.")

# Jika ada, KITA WAJIB MEMBERSIHKANNYA sebelum dikasih ke Anggota 4
df_features_clean = df_features.replace([np.inf, -np.inf], np.nan)
df_features_clean = df_features_clean.fillna(0)

print("\nPembersihan selesai. Data fitur sudah bersih dari inf/NaN.")

In [ ]:
# Gabungkan fitur yang sudah bersih dengan label target
df_combined = pd.concat([df_features_clean.reset_index(drop=True), df['Is_Anomaly'].reset_index(drop=True)], axis=1)

# Hitung korelasi semua fitur terhadap 'Is_Anomaly'
correlation_matrix = df_combined.corr()
target_corr = correlation_matrix['Is_Anomaly'].drop('Is_Anomaly') # Buang korelasi dengan dirinya sendiri

# Ambil 10 fitur dengan korelasi terkuat (baik positif maupun negatif)
top_10_features = target_corr.abs().sort_values(ascending=False).head(10)

# Visualisasi
plt.figure(figsize=(10, 6))
sns.barplot(x=top_10_features.values, y=top_10_features.index, palette='viridis')
plt.title('Top 10 Fitur TSFEL yang Paling Berpengaruh Terhadap Deteksi Anomali')
plt.xlabel('Nilai Korelasi (Absolut)')
plt.ylabel('Nama Fitur TSFEL')
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.show()

### Kesimpulan untuk Anggota 3 & 4:
1. Proses ekstraksi menggunakan TSFEL berjalan lancar dan menghasilkan parameter statistik tambahan.
2. **PERHATIAN:** TSFEL menghasilkan nilai `inf`/`NaN` pada beberapa fitur (seperti variance/kurtosis saat datanya konstan). Logika `fillna(0)` wajib dimasukkan ke dalam skrip produksi (`src/features/build_tsfel.py`).
3. Berdasarkan plot korelasi, beberapa fitur teratas sudah terlihat menjanjikan untuk mendeteksi anomali. Anggota 4 bisa langsung melatih XGBoost dengan fitur-fitur ini!